# SA 1.1 — Euclid Sphere Definition: Lathe Machine Books

**Question:** Which books illustrate *CK_Definition of Sphere by Euclid* exclusively with lathe machine diagrams?


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 2. Load all data


In [ ]:
# ── Image-CK data ───────────────────────────────────────────
df = pd.read_csv('full_image_data_feb_25.csv')

# ── Book-level metadata ──────────────────────────────────────
books = pd.read_csv('full_book_data_feb_25.csv')
df['printer']   = df['book'].map(books.set_index('book')['printers'])
df['publisher'] = df['book'].map(books.set_index('book')['publishers'])

# ── Visual tags (Sphere Definition) ─────────────────────────
VISUAL_TAGS_PATH = ('/Users/nogashlomi/projects/Image_data/'
                    'visual_tags/VT_1.1_sphere_definition.xlsx')
visual_tags = pd.read_excel(VISUAL_TAGS_PATH)

# ── Target CKs ───────────────────────────────────────────────
target_cks = [
    'CK_Definition of Sphere by Theodosius',
    'CK_Definition of Sphere by Euclid',
    'CK_Additions on Definitions of the Sphere',
]

# ── filtered_df: images carrying at least one target CK ──────
images_with_target_cks = df[df['cks'].isin(target_cks)]['images'].unique()
filtered_df = df[df['images'].isin(images_with_target_cks)].copy()

# ── visual_df: filtered_df merged with visual tags ───────────
visual_df = pd.merge(filtered_df, visual_tags, on='cluster_name')

print('✓ Data loaded')
print(f'  df          : {df.shape}')
print(f'  books       : {books.shape}')
print(f'  visual_tags : {visual_tags.shape}')
print(f'  filtered_df : {filtered_df.shape}')
print(f'  visual_df   : {visual_df.shape}')


## 3. Books where ALL Euclid sphere illustrations are lathe machine diagrams

Logic:
- Filter `visual_df` to rows with `CK_Definition of Sphere by Euclid`
- For each book count **total unique images** for that CK and how many have `lathe machine == yes`
- Keep books where those two numbers are equal


In [ ]:
CK_EUCLID = 'CK_Definition of Sphere by Euclid'

# Rows for this CK, one row per (book, image)
euclid_images = (
    visual_df[visual_df['cks'] == CK_EUCLID]
    .drop_duplicates(subset=['bid', 'images'])
    [['bid', 'book', 'images', 'lathe machine',
      'place', 'year', 'printer', 'part_or_adaption_label']]
)

# Per book: total images vs lathe-machine images
book_stats = (
    euclid_images
    .groupby(['bid', 'place', 'year', 'printer'])
    .agg(
        total_euclid_images=('images', 'nunique'),
        lathe_images=('lathe machine',
                       lambda x: (x.str.lower() == 'yes').sum())
    )
    .reset_index()
)

# Books where every Euclid image is a lathe machine
all_lathe_books = book_stats[
    book_stats['total_euclid_images'] == book_stats['lathe_images']
].sort_values('year').reset_index(drop=True)

print(f'Books where ALL Euclid sphere images are lathe machine: {len(all_lathe_books)}')
print(f'(out of {book_stats["bid"].nunique()} books total with this CK)')
all_lathe_books


## 4. Comparing Euclid vs Theodosius: lathe machine books

For each of the two CKs, find books where **all** images are lathe machine diagrams.
Then compare: how many books appear in both groups, only one, etc.


In [ ]:
CK_EUCLID    = 'CK_Definition of Sphere by Euclid'
CK_THEODOSIUS = 'CK_Definition of Sphere by Theodosius'

def lathe_only_books(ck):
    """Return set of bids where ALL images for `ck` are lathe machine."""
    sub = (
        visual_df[visual_df['cks'] == ck]
        .drop_duplicates(subset=['bid', 'images'])
    )
    stats = (
        sub.groupby('bid')
        .agg(
            total=('images', 'nunique'),
            lathe=('lathe machine', lambda x: (x.str.lower() == 'yes').sum())
        )
    )
    return set(stats[stats['total'] == stats['lathe']].index)

def any_books(ck):
    """Return set of ALL bids that have at least one image for `ck`."""
    return set(visual_df[visual_df['cks'] == ck]['bid'].unique())

# ── All books with each CK ────────────────────────────────────────
euclid_all      = any_books(CK_EUCLID)
theodosius_all  = any_books(CK_THEODOSIUS)

# ── Lathe-only books for each CK ─────────────────────────────────
euclid_lathe    = lathe_only_books(CK_EUCLID)
theodosius_lathe = lathe_only_books(CK_THEODOSIUS)

# ── Set comparisons ───────────────────────────────────────────────
both_ck             = euclid_all & theodosius_all          # have both CKs
only_euclid_ck      = euclid_all - theodosius_all          # only Euclid CK
only_theodosius_ck  = theodosius_all - euclid_all          # only Theodosius CK

both_lathe          = euclid_lathe & theodosius_lathe      # lathe-only in both CKs
only_euclid_lathe   = euclid_lathe - theodosius_lathe      # lathe-only for Euclid, not Theodosius
only_theod_lathe    = theodosius_lathe - euclid_lathe      # lathe-only for Theodosius, not Euclid

print('━━━ Books with each CK (any image) ━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Euclid only              : {len(only_euclid_ck):>4}')
print(f'  Theodosius only          : {len(only_theodosius_ck):>4}')
print(f'  Both Euclid & Theodosius : {len(both_ck):>4}')
print()
print('━━━ Of those: books where ALL images for that CK are lathe ━━')
print(f'  Euclid lathe-only        : {len(euclid_lathe):>4}  (of {len(euclid_all)} Euclid books)')
print(f'  Theodosius lathe-only    : {len(theodosius_lathe):>4}  (of {len(theodosius_all)} Theodosius books)')
print()
print('━━━ Overlap between the two lathe-only sets ━━━━━━━━━━━━━━━━━')
print(f'  Lathe-only in BOTH CKs   : {len(both_lathe):>4}')
print(f'  Lathe-only Euclid only   : {len(only_euclid_lathe):>4}')
print(f'  Lathe-only Theodosius only: {len(only_theod_lathe):>3}')


### Book details by group


In [ ]:
# Helper: get book metadata for a set of bids
def book_info(bid_set, label):
    sub = (
        visual_df[visual_df['bid'].isin(bid_set)]
        .drop_duplicates('bid')
        [['bid', 'place', 'year', 'printer', 'part_or_adaption_label']]
        .sort_values('year')
        .reset_index(drop=True)
    )
    print(f'\n── {label} ({len(sub)} books) ──')
    display(sub)

book_info(both_lathe,         'Lathe-only in BOTH Euclid & Theodosius')
book_info(only_euclid_lathe,  'Lathe-only for Euclid ONLY (not Theodosius)')
book_info(only_theod_lathe,   'Lathe-only for Theodosius ONLY (not Euclid)')


### Visual summary


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

# Circles
circle_e = plt.Circle((3.5, 3), 2.2, color='steelblue', alpha=0.35, label='Euclid lathe-only')
circle_t = plt.Circle((6.5, 3), 2.2, color='tomato',    alpha=0.35, label='Theodosius lathe-only')
ax.add_patch(circle_e)
ax.add_patch(circle_t)

# Labels
ax.text(2.2, 3, f'Only Euclid\n{len(only_euclid_lathe)}',
        ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(5.0, 3, f'Both\n{len(both_lathe)}',
        ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(7.8, 3, f'Only Theod.\n{len(only_theod_lathe)}',
        ha='center', va='center', fontsize=13, fontweight='bold')

ax.set_title('Books where ALL sphere-definition images are lathe machine',
             fontsize=12, pad=12)
ax.legend(handles=[
    mpatches.Patch(color='steelblue', alpha=0.5, label=f'Euclid lathe-only ({len(euclid_lathe)} books)'),
    mpatches.Patch(color='tomato',    alpha=0.5, label=f'Theodosius lathe-only ({len(theodosius_lathe)} books)'),
], loc='upper right')
plt.tight_layout()
plt.show()
